In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/263')

Mounted at /content/drive


In [3]:
!pip install -q transformers datasets==2.19.0 accelerate torch
import torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments, BertForMaskedLM, BertTokenizer
from datasets import load_dataset

print("Libraries loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
Libraries loaded successfully.


In [4]:
import torch
import torch.nn.functional as F
from transformers import Trainer

class EmbeddingAwareTrainer(Trainer):
    def __init__(self, *args, lambda_weight=0.0, embed_loss_type='l2', **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_weight = lambda_weight
        self.embed_loss_type = embed_loss_type

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        ce_loss = outputs.get("loss")
        if ce_loss is None:
            loss_fct = torch.nn.CrossEntropyLoss()
            ce_loss = loss_fct(logits.view(-1, model.config.vocab_size), labels.view(-1))

        if self.lambda_weight == 0.0:
            return (ce_loss, outputs) if return_outputs else ce_loss

        mask_idx = labels != -100

        if not mask_idx.any():
            return (ce_loss, outputs) if return_outputs else ce_loss

        masked_logits = logits[mask_idx]
        masked_labels = labels[mask_idx]

        p = F.softmax(masked_logits, dim=-1)
        E = model.get_output_embeddings().weight
        e_hat = torch.matmul(p, E)
        e_gt = E[masked_labels]

        if self.embed_loss_type == 'l2':
            l_embed = F.mse_loss(e_hat, e_gt)
        elif self.embed_loss_type == 'cosine':
            cosine_sim = F.cosine_similarity(e_hat, e_gt, dim=-1)
            l_embed = (1.0 - cosine_sim).mean()
        else:
            raise ValueError("Invalid embed_loss_type")

        total_loss = ce_loss + (self.lambda_weight * l_embed)
        return (total_loss, outputs) if return_outputs else total_loss

print("Trainer class defined.")

Trainer class defined.


In [5]:
#Data Prep
def get_lama_datasets(tokenizer_name='bert-base-uncased', max_length=128):
    tokenizer = BertTokenizer.from_pretrained(tokenizer_name)

    dataset = load_dataset("lama", "trex", split="train[:34000]", trust_remote_code=True)

    def tokenize_function(examples):
        result = tokenizer(examples["masked_sentence"], padding="max_length", truncation=True, max_length=max_length)

        targets = tokenizer(examples["obj_surface"], add_special_tokens=False)

        labels_batch = []

        for i in range(len(result["input_ids"])):
            input_ids = result["input_ids"][i]
            target_ids = targets["input_ids"][i]

            target_id = target_ids[0] if len(target_ids) > 0 else 0

            labels = [-100] * len(input_ids)

            try:
                mask_index = input_ids.index(tokenizer.mask_token_id)
                labels[mask_index] = target_id
            except ValueError:
                pass

            labels_batch.append(labels)

        result["labels"] = labels_batch
        return result

    tokenized_datasets = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    split_dataset = tokenized_datasets.train_test_split(test_size=0.1)

    return split_dataset["train"], split_dataset["test"]

print("Preparing FULL dataset...")
train_dataset, eval_dataset = get_lama_datasets()
print("Data ready: " + str(len(train_dataset)) + " training samples, " + str(len(eval_dataset)) + " eval samples.")

Preparing FULL dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1304391 [00:00<?, ? examples/s]

Map:   0%|          | 0/34000 [00:00<?, ? examples/s]

Data ready: 30600 training samples, 3400 eval samples.


In [22]:
lambda_values = [0.0, 0.1, 0.5, 1.0]
results = {}

for lam in lambda_values:
    print("\n--- Starting run for lambda: " + str(lam) + " ---")

    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    run_name = "lambda_" + str(lam)

    training_args = TrainingArguments(
        output_dir="./results/" + run_name,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        num_train_epochs=3, # adjust?
        per_device_train_batch_size=16,
        logging_steps=10,
        report_to="none"
    )

    trainer = EmbeddingAwareTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        lambda_weight=lam,
        embed_loss_type='l2'
    )

    trainer.train()

    run_key = "run_lambda_" + str(lam)
    results[run_key] = trainer.evaluate()

print("\n=== EXPERIMENT COMPLETE ===")
for key, value in results.items():
    print(key + ": " + str(value))


--- Starting run for lambda: 0.0 ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,0.112386,0.106573
2,0.011276,0.075378
3,0.001817,0.073245


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,0.112855,0.105213
2,0.014618,0.075718
3,0.002750,0.073434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== EXPERIMENT COMPLETE ===
run_lambda_0.0: {'eval_loss': 0.07324469089508057, 'eval_runtime': 32.5441, 'eval_samples_per_second': 104.474, 'eval_steps_per_second': 13.059, 'epoch': 3.0}
run_lambda_0.1: {'eval_loss': 0.07454057037830353, 'eval_runtime': 33.6784, 'eval_samples_per_second': 100.955, 'eval_steps_per_second': 12.619, 'epoch': 3.0}
run_lambda_0.5: {'eval_loss': 0.07330304384231567, 'eval_runtime': 34.0897, 'eval_samples_per_second': 99.737, 'eval_steps_per_second': 12.467, 'epoch': 3.0}
run_lambda_1.0: {'eval_loss': 0.07343367487192154, 'eval_runtime': 34.1116, 'eval_samples_per_second': 99.673, 'eval_steps_per_second': 12.459, 'epoch': 3.0}


In [9]:
#Post training eval
import os
import torch
import numpy as np
from transformers import BertForMaskedLM
from tqdm.auto import tqdm

def get_latest_checkpoint(base_path):
    if not os.path.exists(base_path):
        return None
    checkpoints = [d for d in os.listdir(base_path) if d.startswith("checkpoint")]
    if not checkpoints:
        return base_path
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    return os.path.join(base_path, checkpoints[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_dataloader = torch.utils.data.DataLoader(eval_dataset, batch_size=16)

lambda_values = [0.0, 0.1, 0.5, 1.0]
final_accuracies = {}

print("=== STARTING OFFLINE EVALUATION ===")

for lam in lambda_values:
    base_dir = f"/content/drive/MyDrive/263/results/lambda_{lam}"
    checkpoint_path = get_latest_checkpoint(base_dir)

    if not checkpoint_path:
        print(f"Could not find saved model for lambda {lam}!")
        continue

    print(f"\nLoading lambda = {lam} from {checkpoint_path.split('/')[-1]}...")
    model = BertForMaskedLM.from_pretrained(checkpoint_path).to(device)
    model.eval()

    correct_guesses = 0
    total_masked_tokens = 0

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc=f"Evaluating λ={lam}"):
            input_ids = torch.stack(batch['input_ids']).T.to(device) if isinstance(batch['input_ids'], list) else batch['input_ids'].to(device)
            attention_mask = torch.stack(batch['attention_mask']).T.to(device) if isinstance(batch['attention_mask'], list) else batch['attention_mask'].to(device)
            labels = torch.stack(batch['labels']).T.to(device) if isinstance(batch['labels'], list) else batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            preds = torch.argmax(outputs.logits, dim=-1)

            preds = preds.reshape(-1)
            labels = labels.reshape(-1)

            mask = labels != -100
            valid_preds = preds[mask]
            valid_labels = labels[mask]

            correct_guesses += (valid_preds == valid_labels).sum().item()
            total_masked_tokens += mask.sum().item()

    accuracy = correct_guesses / total_masked_tokens
    final_accuracies[f"λ={lam}"] = accuracy

print("\n=== FINAL FACTUAL RECALL ACCURACY ===")
for key, acc in final_accuracies.items():
    print(f"{key}: {acc:.2%}")

=== STARTING OFFLINE EVALUATION ===

Loading lambda = 0.0 from checkpoint-5739...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=0.0:   0%|          | 0/213 [00:00<?, ?it/s]


Loading lambda = 0.1 from checkpoint-5739...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Evaluating λ=0.5:   0%|          | 0/213 [00:00<?, ?it/s]


Loading lambda = 1.0 from checkpoint-5739...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=1.0:   0%|          | 0/213 [00:00<?, ?it/s]


=== FINAL FACTUAL RECALL ACCURACY ===
λ=0.0: 99.71%
λ=0.1: 99.71%
λ=0.5: 99.74%
λ=1.0: 99.74%


500 Row Experiment


In [5]:
#Data Prep, smaller 500 training
def get_lama_datasets(tokenizer_name='bert-base-uncased', max_length=128):
    tokenizer = BertTokenizer.from_pretrained(tokenizer_name)

    dataset = load_dataset("lama", "trex", split="train[:500]", trust_remote_code=True)

    def tokenize_function(examples):
        result = tokenizer(examples["masked_sentence"], padding="max_length", truncation=True, max_length=max_length)

        targets = tokenizer(examples["obj_surface"], add_special_tokens=False)

        labels_batch = []

        for i in range(len(result["input_ids"])):
            input_ids = result["input_ids"][i]
            target_ids = targets["input_ids"][i]

            target_id = target_ids[0] if len(target_ids) > 0 else 0

            labels = [-100] * len(input_ids)

            try:
                mask_index = input_ids.index(tokenizer.mask_token_id)
                labels[mask_index] = target_id
            except ValueError:
                pass

            labels_batch.append(labels)

        result["labels"] = labels_batch
        return result

    tokenized_datasets = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    split_dataset = tokenized_datasets.train_test_split(test_size=0.1)

    return split_dataset["train"], split_dataset["test"]

print("Preparing FULL dataset...")
train_dataset, eval_dataset = get_lama_datasets()
print("Data ready: " + str(len(train_dataset)) + " training samples, " + str(len(eval_dataset)) + " eval samples.")

Preparing FULL dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1304391 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Data ready: 450 training samples, 50 eval samples.


In [6]:
lambda_values = [0.0, 0.1, 0.5, 1.0]
results = {}

for lam in lambda_values:
    print("\n--- Starting run for lambda: " + str(lam) + " ---")

    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    run_name = "lambda_" + str(lam)

    training_args = TrainingArguments(
        output_dir="./results2/" + run_name,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        num_train_epochs=3, # adjust?
        per_device_train_batch_size=16,
        logging_steps=10,
        report_to="none"
    )

    trainer = EmbeddingAwareTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        lambda_weight=lam,
        embed_loss_type='l2'
    )

    trainer.train()

    run_key = "run_lambda_" + str(lam)
    results[run_key] = trainer.evaluate()

print("\n=== EXPERIMENT COMPLETE ===")
for key, value in results.items():
    print(key + ": " + str(value))


--- Starting run for lambda: 0.0 ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Epoch,Training Loss,Validation Loss
1,0.342277,0.437321
2,0.022202,0.385668
3,0.001995,0.387991


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== EXPERIMENT COMPLETE ===
run_lambda_0.0: {'eval_loss': 0.3879227340221405, 'eval_runtime': 0.5613, 'eval_samples_per_second': 89.072, 'eval_steps_per_second': 12.47, 'epoch': 3.0}
run_lambda_0.1: {'eval_loss': 0.3879290521144867, 'eval_runtime': 0.6088, 'eval_samples_per_second': 82.129, 'eval_steps_per_second': 11.498, 'epoch': 3.0}
run_lambda_0.5: {'eval_loss': 0.3879568874835968, 'eval_runtime': 0.5783, 'eval_samples_per_second': 86.455, 'eval_steps_per_second': 12.104, 'epoch': 3.0}
run_lambda_1.0: {'eval_loss': 0.38799118995666504, 'eval_runtime': 0.6084, 'eval_samples_per_second': 82.189, 'eval_steps_per_second': 11.506, 'epoch': 3.0}


In [7]:
#Post training eval
import os
import torch
import numpy as np
from transformers import BertForMaskedLM
from tqdm.auto import tqdm

def get_latest_checkpoint(base_path):
    if not os.path.exists(base_path):
        return None
    checkpoints = [d for d in os.listdir(base_path) if d.startswith("checkpoint")]
    if not checkpoints:
        return base_path
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    return os.path.join(base_path, checkpoints[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_dataloader = torch.utils.data.DataLoader(eval_dataset, batch_size=16)

lambda_values = [0.0, 0.1, 0.5, 1.0]
final_accuracies = {}

print("=== STARTING OFFLINE EVALUATION ===")

for lam in lambda_values:
    base_dir = f"/content/drive/MyDrive/263/results2/lambda_{lam}"
    checkpoint_path = get_latest_checkpoint(base_dir)

    if not checkpoint_path:
        print(f"Could not find saved model for lambda {lam}!")
        continue

    print(f"\nLoading lambda = {lam} from {checkpoint_path.split('/')[-1]}...")
    model = BertForMaskedLM.from_pretrained(checkpoint_path).to(device)
    model.eval()

    correct_guesses = 0
    total_masked_tokens = 0

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc=f"Evaluating λ={lam}"):
            input_ids = torch.stack(batch['input_ids']).T.to(device) if isinstance(batch['input_ids'], list) else batch['input_ids'].to(device)
            attention_mask = torch.stack(batch['attention_mask']).T.to(device) if isinstance(batch['attention_mask'], list) else batch['attention_mask'].to(device)
            labels = torch.stack(batch['labels']).T.to(device) if isinstance(batch['labels'], list) else batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            preds = torch.argmax(outputs.logits, dim=-1)

            preds = preds.reshape(-1)
            labels = labels.reshape(-1)

            mask = labels != -100
            valid_preds = preds[mask]
            valid_labels = labels[mask]

            correct_guesses += (valid_preds == valid_labels).sum().item()
            total_masked_tokens += mask.sum().item()

    accuracy = correct_guesses / total_masked_tokens
    final_accuracies[f"λ={lam}"] = accuracy

print("\n=== FINAL FACTUAL RECALL ACCURACY ===")
for key, acc in final_accuracies.items():
    print(f"{key}: {acc:.2%}")

=== STARTING OFFLINE EVALUATION ===

Loading lambda = 0.0 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=0.0:   0%|          | 0/4 [00:00<?, ?it/s]


Loading lambda = 0.1 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Evaluating λ=0.5:   0%|          | 0/4 [00:00<?, ?it/s]


Loading lambda = 1.0 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=1.0:   0%|          | 0/4 [00:00<?, ?it/s]


=== FINAL FACTUAL RECALL ACCURACY ===
λ=0.0: 98.00%
λ=0.1: 98.00%
λ=0.5: 98.00%
λ=1.0: 98.00%


In [8]:
lambda_values = [0.0, 100.0, 1000.0, 10000.0]
results = {}

for lam in lambda_values:
    print("\n--- Starting run for lambda: " + str(lam) + " ---")

    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    run_name = "lambda_" + str(lam)

    training_args = TrainingArguments(
        output_dir="./results3/" + run_name,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        num_train_epochs=3, # adjust?
        per_device_train_batch_size=16,
        logging_steps=10,
        report_to="none"
    )

    trainer = EmbeddingAwareTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        lambda_weight=lam,
        embed_loss_type='l2'
    )

    trainer.train()

    run_key = "run_lambda_" + str(lam)
    results[run_key] = trainer.evaluate()

print("\n=== EXPERIMENT COMPLETE ===")
for key, value in results.items():
    print(key + ": " + str(value))


--- Starting run for lambda: 0.0 ---


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss
1,0.342186,0.437256
2,0.022188,0.385605
3,0.001995,0.387923


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,1.651417,1.161511
2,0.553094,0.994832
3,0.127114,0.994154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== EXPERIMENT COMPLETE ===
run_lambda_0.0: {'eval_loss': 0.3879227340221405, 'eval_runtime': 0.5757, 'eval_samples_per_second': 86.844, 'eval_steps_per_second': 12.158, 'epoch': 3.0}
run_lambda_100.0: {'eval_loss': 0.39480048418045044, 'eval_runtime': 0.6008, 'eval_samples_per_second': 83.224, 'eval_steps_per_second': 11.651, 'epoch': 3.0}
run_lambda_1000.0: {'eval_loss': 0.4516584873199463, 'eval_runtime': 0.641, 'eval_samples_per_second': 78.001, 'eval_steps_per_second': 10.92, 'epoch': 3.0}
run_lambda_10000.0: {'eval_loss': 0.9941543340682983, 'eval_runtime': 0.5807, 'eval_samples_per_second': 86.107, 'eval_steps_per_second': 12.055, 'epoch': 3.0}


In [10]:
#Post training eval
import os
import torch
import numpy as np
from transformers import BertForMaskedLM
from tqdm.auto import tqdm

def get_latest_checkpoint(base_path):
    if not os.path.exists(base_path):
        return None
    checkpoints = [d for d in os.listdir(base_path) if d.startswith("checkpoint")]
    if not checkpoints:
        return base_path
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    return os.path.join(base_path, checkpoints[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_dataloader = torch.utils.data.DataLoader(eval_dataset, batch_size=16)

lambda_values = [0.0, 100.0, 1000.0, 10000.0]
final_accuracies = {}

print("=== STARTING OFFLINE EVALUATION ===")

for lam in lambda_values:
    base_dir = f"/content/drive/MyDrive/263/results3/lambda_{lam}"
    checkpoint_path = get_latest_checkpoint(base_dir)

    if not checkpoint_path:
        print(f"Could not find saved model for lambda {lam}!")
        continue

    print(f"\nLoading lambda = {lam} from {checkpoint_path.split('/')[-1]}...")
    model = BertForMaskedLM.from_pretrained(checkpoint_path).to(device)
    model.eval()

    correct_guesses = 0
    total_masked_tokens = 0

    with torch.no_grad():
        for batch in tqdm(eval_dataloader, desc=f"Evaluating λ={lam}"):
            input_ids = torch.stack(batch['input_ids']).T.to(device) if isinstance(batch['input_ids'], list) else batch['input_ids'].to(device)
            attention_mask = torch.stack(batch['attention_mask']).T.to(device) if isinstance(batch['attention_mask'], list) else batch['attention_mask'].to(device)
            labels = torch.stack(batch['labels']).T.to(device) if isinstance(batch['labels'], list) else batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            preds = torch.argmax(outputs.logits, dim=-1)

            preds = preds.reshape(-1)
            labels = labels.reshape(-1)

            mask = labels != -100
            valid_preds = preds[mask]
            valid_labels = labels[mask]

            correct_guesses += (valid_preds == valid_labels).sum().item()
            total_masked_tokens += mask.sum().item()

    accuracy = correct_guesses / total_masked_tokens
    final_accuracies[f"λ={lam}"] = accuracy

print("\n=== FINAL FACTUAL RECALL ACCURACY ===")
for key, acc in final_accuracies.items():
    print(f"{key}: {acc:.2%}")

=== STARTING OFFLINE EVALUATION ===

Loading lambda = 0.0 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=0.0:   0%|          | 0/4 [00:00<?, ?it/s]


Loading lambda = 100.0 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Evaluating λ=1000.0:   0%|          | 0/4 [00:00<?, ?it/s]


Loading lambda = 10000.0 from checkpoint-87...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating λ=10000.0:   0%|          | 0/4 [00:00<?, ?it/s]


=== FINAL FACTUAL RECALL ACCURACY ===
λ=0.0: 98.00%
λ=100.0: 98.00%
λ=1000.0: 98.00%
λ=10000.0: 98.00%
